# Do bucket a gold — por que a tabela precisa de ETLEquivale aos scripts `01_auditoria_silver.sql` e `02_gold_etl.sql`.---**PDM 2026.2 — aula de 04/09/2026 · notebook 1 de 3**Este notebook e o mesmo conteudo dos scripts `.sql` da aula, so que rodando de dentrodo Python. Serve para estudar sozinho, para repetir a aula no seu ritmo e como pontepara a proxima aula, em que o dado sai do BigQuery e vira DataFrame.Voce pode rodar tudo no [Google Colab](https://colab.research.google.com) sem instalar nada.Se rodar na sua maquina, precisa de `pip install google-cloud-bigquery pandas db-dtypes`e de um `gcloud auth application-default login` antes.Uma coisa de cada vez: **nao rode tudo de uma vez**. Rode uma celula, leia o resultado,so entao passe para a proxima. O valor da aula esta em olhar para o numero que aparece.

In [ ]:
# Rode este bloco UMA VEZ, antes de qualquer outro.# Ele autentica voce no Google Cloud e cria os atalhos q(...) e run(...).PROJETO = "SEU_PROJETO"      # <<< troque pelo id do SEU projeto no GCPDATASET = "anuncios"LOCAL   = "us-central1"      # a regiao do dataset. Precisa bater, senao o BigQuery recusa.try:    from google.colab import auth    auth.authenticate_user()          # Colab: abre a janela de login do Googleexcept ImportError:    pass                              # local: rode antes `gcloud auth application-default login`import pandas as pdfrom google.cloud import bigqueryclient = bigquery.Client(project=PROJETO, location=LOCAL)def q(sql: str) -> pd.DataFrame:    """Roda o SQL e devolve um DataFrame. Toda ocorrencia de SEU_PROJETO vira o seu projeto."""    return client.query(sql.replace("SEU_PROJETO", PROJETO)).to_dataframe()def run(sql: str) -> None:    """Para CREATE TABLE / CREATE MODEL: executa, nao devolve tabela."""    client.query(sql.replace("SEU_PROJETO", PROJETO)).result()    print("ok")pd.set_option("display.float_format", lambda v: f"{v:,.2f}")print("conectado em", PROJETO, "|", LOCAL)

## 1. Auditoria da silverA `tb_anuncios_silver` tem 1.000 anuncios de imoveis de Goiania. Ela ja esta tipada:`preco` e numero, `area_util` e numero, `bairro` e texto. Parece pronta.Comece pelo perfil do alvo. Tres numeros bastam para desconfiar.

In [ ]:
q("""SELECT  COUNT(*)                                        AS anuncios,  ROUND(MIN(preco))                               AS menor_preco,  ROUND(APPROX_QUANTILES(preco, 100)[OFFSET(50)]) AS mediana,  ROUND(AVG(preco))                               AS media,  ROUND(MAX(preco))                               AS maior_preco,  ROUND(STDDEV(preco))                            AS desvio_padraoFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE preco IS NOT NULL""")

**Como ler.** A media e varias vezes maior que a mediana. Isso nunca acontece poracaso: significa que um punhado de valores gigantes esta puxando a media para cima.E o desvio padrao maior que a propria media diz que a nuvem de pontos e larga demaispara qualquer reta passar no meio dela.Quem sao os gigantes?

In [ ]:
q("""SELECT bairro, titulo, area_util, quartos, ROUND(preco) AS precoFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE preco > 50000000ORDER BY preco DESCLIMIT 10""")

**Como ler.** Olhe o titulo ao lado do preco. Um apartamento de 3 quartos anunciadopor centenas de milhoes nao e um imovel caro: e um erro de digitacao ou de escala(alguem digitou o valor em centavos, ou repetiu zeros). O dado nao esta "errado nosistema", esta errado no mundo.Quanto eles custam para o modelo?

In [ ]:
q("""SELECT  COUNTIF(preco > 50000000)                            AS anuncios_suspeitos,  COUNT(*)                                             AS total,  ROUND(100 * COUNTIF(preco > 50000000) / COUNT(*), 1) AS pct,  ROUND(STDDEV(preco))                                 AS desvio_com_eles,  ROUND(STDDEV(IF(preco <= 50000000, preco, NULL)))    AS desvio_sem_elesFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE preco IS NOT NULL""")

**Como ler.** Compare `desvio_com_eles` com `desvio_sem_eles`. Uma fracao minusculadas linhas responde pela maior parte da dispersao. Este e o argumento inteiro do ETLem dois numeros: voce joga fora quase nada de dado e recupera quase toda a estrutura.O mesmo vale para a area.

In [ ]:
q("""SELECT bairro, titulo, area_util, area_total, ROUND(preco) AS precoFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE area_util > 100000ORDER BY area_util DESCLIMIT 10""")

In [ ]:
q("""SELECT  COUNTIF(area_util > 10000) AS acima_de_10mil_m2,  COUNTIF(area_util <= 1)    AS area_zero_ou_um,  COUNT(*)                   AS totalFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`""")

**Como ler.** Area util acima de 10.000 m2 quase sempre e hectare digitado comometro quadrado. Area 0 ou 1 e campo que ninguem preencheu. Sao dois defeitosopostos que o mesmo filtro resolve: um intervalo plausivel.Agora as colunas que nao servem para nada.

In [ ]:
q("""SELECT  COUNT(DISTINCT tipo_contrato) AS tipos_contrato,  COUNT(DISTINCT status)        AS status_distintos,  COUNT(DISTINCT cidade)        AS cidadesFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`""")

**Como ler.** Coluna com um unico valor distinto nao carrega informacao nenhuma:se todo mundo tem o mesmo valor, ela nao ajuda a separar caro de barato. Nao euma coluna suja, e uma coluna morta. Ela sai da gold por inutilidade, nao por erro.E o defeito mais silencioso de todos.

In [ ]:
q("""SELECT id_anuncio, bairro, latitude, longitudeFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE latitude < -40 OR latitude IS NULLORDER BY latitude NULLS LASTLIMIT 10""")

**Como ler.** Goiania fica perto de latitude -16 e longitude -49. Onde a latitudeaparece como -49, alguem trocou os dois campos de lugar. O tipo esta certo, ointervalo esta plausivel para *um* dos campos, e mesmo assim o dado esta errado.Nenhuma validacao de schema pega isso. So pega quem conhece o dominio.> `NULLS LAST` importa: sem ele, os NULLs vem primeiro e escondem as linhas trocadas.---## 2. Construir a goldA gold nao e "a silver limpa". E uma tabela **feita para uma pergunta**: prever preco.Tudo que nao ajuda nessa pergunta sai; o que falta, a gente cria.Primeiro, uma coluna que a silver nao tem: o imovel e comercial?

In [ ]:
q(r"""SELECT titulo, ROUND(preco) AS precoFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE REGEXP_CONTAINS(        LOWER(titulo),        r'comercial|galp[aã]o|pr[eé]dio|loja|escrit[oó]rio|barrac[aã]o|industrial')LIMIT 10""")

In [ ]:
q(r"""SELECT  COUNTIF(REGEXP_CONTAINS(LOWER(titulo),    r'comercial|galp[aã]o|pr[eé]dio|loja|escrit[oó]rio|barrac[aã]o|industrial'))    AS comerciais,  COUNT(*) AS totalFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`""")

**Como ler.** O titulo e texto livre, o campo mais bagunçado da tabela — e mesmoassim carrega uma informacao estrutural que nenhuma coluna numerica tem. Um galpaoe um apartamento tem precos que obedecem a logicas diferentes. Dizer isso ao modeloe o que vamos chamar de *engenharia de feature*.Guarde essa frase, porque no notebook 3 a mesma coluna e a mesma funcao vao produziruma fraude.Agora a gold inteira.

In [ ]:
run(r"""CREATE OR REPLACE TABLE `SEU_PROJETO.anuncios.anuncios_gold` ASSELECT  id_anuncio,  preco,  area_util,  quartos,  banheiros,  suites,  garagem,  condominio,  iptu,  bairro,  CASE WHEN latitude < -40 THEN longitude ELSE latitude  END AS latitude,  CASE WHEN latitude < -40 THEN latitude  ELSE longitude END AS longitude,  REGEXP_CONTAINS(LOWER(titulo),    r'comercial|galp[aã]o|pr[eé]dio|loja|escrit[oó]rio|barrac[aã]o|industrial')    AS eh_comercial,  ROUND(preco / area_util, 2) AS preco_por_m2,  tituloFROM `SEU_PROJETO.anuncios.tb_anuncios_silver`WHERE      preco IS NOT NULL  AND preco BETWEEN 50000 AND 50000000  AND area_util BETWEEN 20 AND 10000""")

**Como ler o CREATE.** Ele faz quatro coisas ao mesmo tempo, e vale separar:1. **corrige** — o `CASE WHEN latitude < -40` desfaz a troca de coordenadas;2. **cria** — `eh_comercial` e `preco_por_m2` nao existiam;3. **descarta** — `tipo_contrato`, `status` e `cidade` simplesmente nao aparecem no SELECT;4. **filtra** — o `WHERE` corta as linhas impossiveis.Nenhuma dessas quatro e uma decisao do algoritmo. Todas sao suas.

In [ ]:
q("""SELECT  COUNT(*)                                        AS anuncios,  ROUND(APPROX_QUANTILES(preco, 100)[OFFSET(50)]) AS mediana,  ROUND(AVG(preco))                               AS media,  ROUND(STDDEV(preco))                            AS desvio,  COUNTIF(eh_comercial)                           AS comerciais,  ROUND(100 * COUNTIF(eh_comercial) / COUNT(*), 1) AS pct_comercialFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

**Como ler.** Compare com o primeiro perfil deste notebook. Media e mediana agoraestao na mesma ordem de grandeza, e o desvio caiu para um valor que faz sentido paraimoveis de Goiania. Sobraram 887 das 1.000 linhas: voce perdeu 11% do dado e ganhouuma tabela em que uma reta consegue passar.---## O que fica- Tipagem correta nao e qualidade de dado. A silver estava tipada e ainda assim devia limpeza.- Um punhado de linhas absurdas domina qualquer estatistica agregada.- Coluna de valor unico e peso morto, nao e sujeira.- O erro mais caro (latitude/longitude trocadas) e invisivel para validacao automatica.**Proximo notebook:** treinar o primeiro modelo sobre esta gold e descobrir quantoesse ETL valeu, em reais de erro.